# Segmentation Pipeline v1 — GT-bbox Crop + Cls Retrain + Lightweight Detect

**Hedef:** Whole-image classification yerine **two-stage pipeline (detect → crop → classify)** kullanarak hem test seti doğruluğunu hem de mobil gerçek-dünya doğruluğunu arttırmak.

**Strateji:**
1. **Phase 1**: Ground-truth bbox'larla 25.915 görüntüyü kropla (eğitim için temiz veri)
2. **Phase 2**: YOLO11m-cls'i kırpılmış veride sıfırdan yeniden eğit (50 epoch)
3. **Phase 3**: Original (whole-image) cls vs retrained (cropped) cls karşılaştır + McNemar
4. **Phase 4**: Mobile için YOLO11n-detect eğit (~6 MB, lightweight)
5. **Phase 5**: Her iki modeli FP16 ONNX export

**Beklenti:**
- Test seti: +%2-4 accuracy, +%3-5 mel recall (lezyon-fokus etkisi)
- Mobil gerçek-dünya: +%5-10 effective accuracy (off-center/gürültülü fotoğraflarda)

**Süre:** ~3-4 saat (A100), Phase 2 + Phase 4 ana yük.

## 0. Ortam

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U ultralytics onnx onnxconverter-common onnxruntime

import subprocess, sys
check = subprocess.run([sys.executable, '-c', 'import torch; exit(0 if torch.cuda.is_available() else 1)'])
if check.returncode != 0:
    print('⚠ CUDA torch kaybolmuş, geri yükleniyor...')
    !pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu128
    print('Runtime → Restart session yapın!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, shutil, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image

PROJECT_ROOT  = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProje'
DRIVE_DATA    = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProjesi'
LOCAL_ROOT    = '/content/ciltkanseri'
IMG_ROOT      = f'{LOCAL_ROOT}/dataset_yolo/images'
LBL_ROOT      = f'{LOCAL_ROOT}/dataset_yolo/labels'
CROPPED_ROOT  = f'{LOCAL_ROOT}/cls_data_cropped'  # yeni kırpılmış veri
YOLO_RUNS     = f'{PROJECT_ROOT}/yolo_runs'

os.makedirs(CROPPED_ROOT, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')

CLASS_NAMES = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

# Veri yoksa Drive zip'inden aç
if not os.path.exists(f'{IMG_ROOT}/train'):
    print('Veri açılıyor...')
    !unzip -q "$DRIVE_DATA/dataset_yolo.zip" -d $LOCAL_ROOT
else:
    print('Veri zaten açık.')

# Sayım kontrol
for sp in ['train', 'val', 'test']:
    img_count = len(glob.glob(f'{IMG_ROOT}/{sp}/*.jpg'))
    lbl_count = len(glob.glob(f'{LBL_ROOT}/{sp}/*.txt'))
    print(f'  {sp}: {img_count} görüntü, {lbl_count} label dosyası')

## Phase 1 — GT-bbox ile Kropla

Her görüntü için label dosyasından bbox'ı oku, %20 margin ekle, 384×384'e karelliyerek (square padding ile) kaydet. Sınıf-folder formatında YOLO classification için hazırla.

In [ ]:
from PIL import Image

def crop_with_margin(img, bbox_yolo, margin=0.2, target=384):
    """YOLO format bbox (cx, cy, w, h normalized) → cropped square image (target x target).
    
    Args:
        img: PIL Image
        bbox_yolo: tuple (cx, cy, w, h) all in [0, 1] normalized coords
        margin: bbox boyutuna ek padding (0.2 = %20)
        target: çıktı boyutu (384x384)
    """
    W, H = img.size
    cx, cy, w, h = bbox_yolo
    # Margin ekle
    w *= (1 + margin)
    h *= (1 + margin)
    # Square crop için en uzun kenarı al
    side = max(w * W, h * H)
    # Pikselleri
    px_cx = cx * W
    px_cy = cy * H
    half = side / 2
    # Crop koordinatları
    left = max(0, px_cx - half)
    top  = max(0, px_cy - half)
    right = min(W, px_cx + half)
    bottom = min(H, px_cy + half)
    # Crop
    cropped = img.crop((left, top, right, bottom))
    # Square padding (eğer kenar resmin dışına taştıysa)
    cw, ch = cropped.size
    if cw != ch:
        s = max(cw, ch)
        sq = Image.new('RGB', (s, s), (128, 128, 128))  # gri padding
        sq.paste(cropped, ((s - cw) // 2, (s - ch) // 2))
        cropped = sq
    # Resize to target
    cropped = cropped.resize((target, target), Image.LANCZOS)
    return cropped

def parse_label_file(label_path):
    """YOLO label dosyasını oku, ilk bbox'ı döndür (genelde tek lezyon var)."""
    with open(label_path) as f:
        lines = [l.strip().split() for l in f if l.strip()]
    if not lines:
        return None
    parts = lines[0]
    cls_id = int(parts[0])
    cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    return cls_id, (cx, cy, w, h)

# Class folder yapısı oluştur
for sp in ['train', 'val', 'test']:
    for cls in CLASS_NAMES:
        os.makedirs(f'{CROPPED_ROOT}/{sp}/{cls}', exist_ok=True)

# Tüm görüntüleri kropla
stats = {sp: {'ok': 0, 'no_label': 0, 'error': 0} for sp in ['train', 'val', 'test']}
t0 = time.time()

for sp in ['train', 'val', 'test']:
    img_files = sorted(glob.glob(f'{IMG_ROOT}/{sp}/*.jpg'))
    print(f'\n=== {sp.upper()}: {len(img_files)} görüntü ===')
    for i, img_path in enumerate(img_files):
        name = os.path.basename(img_path)
        # Label
        label_path = f'{LBL_ROOT}/{sp}/{os.path.splitext(name)[0]}.txt'
        if not os.path.exists(label_path):
            stats[sp]['no_label'] += 1
            continue
        try:
            label = parse_label_file(label_path)
            if label is None:
                stats[sp]['no_label'] += 1
                continue
            cls_id, bbox = label
            cls_name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else 'nv'
            
            img = Image.open(img_path).convert('RGB')
            cropped = crop_with_margin(img, bbox, margin=0.20, target=384)
            
            # Dosya ismi: prefix sınıf adı (yolo cls otomatik anlasın)
            out_path = f'{CROPPED_ROOT}/{sp}/{cls_name}/{name}'
            cropped.save(out_path, quality=92)
            stats[sp]['ok'] += 1
        except Exception as e:
            stats[sp]['error'] += 1
            if stats[sp]['error'] < 3:
                print(f'  HATA {name}: {e}')
        
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(img_files)} | elapsed {time.time()-t0:.0f}s')

print(f'\n=== KROPLAMA ÖZET ({(time.time()-t0)/60:.1f} dk) ===')
for sp, s in stats.items():
    print(f'  {sp}: ✅ {s["ok"]}, ⚠ no_label {s["no_label"]}, ❌ error {s["error"]}')

# Sınıf dağılımı kontrol
print('\n=== KIRPILMIŞ VERİ SINIF DAĞILIMI ===')
for sp in ['train', 'val', 'test']:
    counts = {cls: len(os.listdir(f'{CROPPED_ROOT}/{sp}/{cls}')) for cls in CLASS_NAMES}
    print(f'  {sp}: {counts}')

## Phase 1.5 — Train için oversampling (symlink, mevcut yöntemle aynı)

In [ ]:
OVERSAMPLE_TARGET = 3000  # her sınıf için (azınlık sınıfları)

TRAIN_BALANCED = f'{LOCAL_ROOT}/cls_data_cropped_balanced/train'
VAL_BALANCED   = f'{LOCAL_ROOT}/cls_data_cropped_balanced/val'
TEST_BALANCED  = f'{LOCAL_ROOT}/cls_data_cropped_balanced/test'

for sp_dst, sp_src in [(TRAIN_BALANCED, 'train'), (VAL_BALANCED, 'val'), (TEST_BALANCED, 'test')]:
    if os.path.exists(sp_dst):
        shutil.rmtree(sp_dst)
    os.makedirs(sp_dst, exist_ok=True)
    for cls in CLASS_NAMES:
        os.makedirs(f'{sp_dst}/{cls}', exist_ok=True)
        src_files = glob.glob(f'{CROPPED_ROOT}/{sp_src}/{cls}/*.jpg')
        
        # Train için oversample (symlink), val/test için orijinal
        if sp_src == 'train' and len(src_files) < OVERSAMPLE_TARGET and len(src_files) > 0:
            target_n = OVERSAMPLE_TARGET
        else:
            target_n = len(src_files)
        
        for i in range(target_n):
            src = src_files[i % len(src_files)]
            dst = f'{sp_dst}/{cls}/{i:05d}_{os.path.basename(src)}'
            if not os.path.exists(dst):
                os.symlink(src, dst)
    
    print(f'{sp_dst}:')
    for cls in CLASS_NAMES:
        print(f'  {cls}: {len(os.listdir(f"{sp_dst}/{cls}"))}')

CLS_ROOT_BALANCED = f'{LOCAL_ROOT}/cls_data_cropped_balanced'
print(f'\n✅ Balanced dataset hazır: {CLS_ROOT_BALANCED}')

## Phase 2 — YOLO11m-cls'i kırpılmış veride yeniden eğit

In [ ]:
from ultralytics import YOLO

CFG = {
    'epochs': 50,
    'img_size': 384,
    'batch_size': 128,
    'patience': 15,
    'lr0': 0.001,
    'lrf': 0.01,
}

RUN_NAME = 'yolo11m_cls_cropped'
RUN_DIR = f'{YOLO_RUNS}/{RUN_NAME}'

if os.path.exists(f'{RUN_DIR}/weights/best.pt'):
    print(f'⏩ Model zaten eğitilmiş: {RUN_DIR}/weights/best.pt — atlanıyor')
    print('Yeniden eğitmek için klasörü silin: !rm -rf $RUN_DIR')
else:
    print('=' * 60)
    print(f'YOLO11m-cls (CROPPED) EĞİTİM BAŞLIYOR')
    print('=' * 60)
    
    t0 = time.time()
    model = YOLO('yolo11m-cls.pt')
    _ = model.train(
        data=CLS_ROOT_BALANCED,
        epochs=CFG['epochs'],
        imgsz=CFG['img_size'],
        batch=CFG['batch_size'],
        patience=CFG['patience'],
        device=0,
        amp=True,
        optimizer='AdamW',
        lr0=CFG['lr0'],
        lrf=CFG['lrf'],
        cos_lr=True,
        # Augmentation (cropped data için biraz daha hafif)
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=20, translate=0.1, scale=0.4,
        fliplr=0.5, flipud=0.5,
        mosaic=0.0,  # cropped data için mosaic kapalı (lezyon zaten merkezde)
        mixup=0.05,
        erasing=0.2,
        # Sistem
        project=YOLO_RUNS,
        name=RUN_NAME,
        exist_ok=True,
        verbose=True,
    )
    elapsed = (time.time() - t0) / 60
    print(f'\n✅ Eğitim tamamlandı: {elapsed:.1f} dk')

## Phase 3 — Test seti karşılaştırması: Cropped cls vs Original cls

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    f1_score, accuracy_score, recall_score, balanced_accuracy_score
)
from statsmodels.stats.contingency_tables import mcnemar

ULTRA_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
ultra2our = [label2idx[c] for c in ULTRA_ORDER]

def eval_yolo_cls(model_pt, test_dir, img_size=384):
    """Test setinde YOLO cls modelini değerlendirir."""
    m = YOLO(model_pt)
    test_files = []
    test_labels = []
    for cls in CLASS_NAMES:
        files = glob.glob(f'{test_dir}/{cls}/*.jpg')
        test_files.extend(files)
        test_labels.extend([label2idx[cls]] * len(files))
    
    y_pred = []
    probs = []
    for bs in range(0, len(test_files), 64):
        batch = test_files[bs:bs+64]
        results = m.predict(batch, imgsz=img_size, device=0, verbose=False)
        for r in results:
            p_u = r.probs.data.cpu().numpy()
            p_o = np.zeros(7)
            for i, oi in enumerate(ultra2our):
                p_o[oi] = p_u[i]
            probs.append(p_o)
            y_pred.append(p_o.argmax())
    
    y_true = np.array(test_labels)
    y_pred = np.array(y_pred)
    probs = np.array(probs)
    
    metrics = {
        'accuracy'    : accuracy_score(y_true, y_pred),
        'bal_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1'    : f1_score(y_true, y_pred, average='macro'),
        'mel_recall'  : recall_score(y_true, y_pred, labels=[label2idx['mel']], average='macro'),
        'mel_precision': ((y_pred == label2idx['mel']) & (y_true == label2idx['mel'])).sum() / max(1, (y_pred == label2idx['mel']).sum()),
        'kappa'       : cohen_kappa_score(y_true, y_pred),
    }
    return metrics, y_true, y_pred, probs

# Original cls (whole-image test set)
ORIG_CLS_PT = f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt'
CROP_CLS_PT = f'{RUN_DIR}/weights/best.pt'

# IMPORTANT: orig cls'i ORİGİNAL test set üzerinde değerlendir
# crop cls'i CROPPED test set üzerinde değerlendir
# Her ikisi aynı görüntülerin farklı versiyonları (cropped vs whole)

ORIG_TEST = f'{LOCAL_ROOT}/cls_data/test'
CROP_TEST = f'{CROPPED_ROOT}/test'

# Eğer orig cls_data yoksa (önceki notebook kurmuş olmalı), hızlıca kur
if not os.path.exists(ORIG_TEST):
    print('Original cls_data yok, oluşturuluyor...')
    os.makedirs(ORIG_TEST, exist_ok=True)
    for cls in CLASS_NAMES:
        os.makedirs(f'{ORIG_TEST}/{cls}', exist_ok=True)
    for img_path in glob.glob(f'{IMG_ROOT}/test/*.jpg'):
        name = os.path.basename(img_path)
        cls = name.split('_', 1)[0]
        if cls in CLASS_NAMES:
            os.symlink(img_path, f'{ORIG_TEST}/{cls}/{name}')

print('=== ORIGINAL CLS (whole-image) test ===')
orig_m, y_true, y_pred_orig, probs_orig = eval_yolo_cls(ORIG_CLS_PT, ORIG_TEST)
for k, v in orig_m.items():
    print(f'  {k:15s}: {v:.4f}')

print('\n=== CROPPED CLS test ===')
crop_m, y_true2, y_pred_crop, probs_crop = eval_yolo_cls(CROP_CLS_PT, CROP_TEST)
for k, v in crop_m.items():
    print(f'  {k:15s}: {v:.4f}')

# DİKKAT: y_true ve y_true2 aynı sıraya sahip mi kontrol et
# Eğer test_files sıralaması aynı olursa y_true == y_true2 olmalı
print(f'\ny_true eşit mi: {np.array_equal(y_true, y_true2)}')
if not np.array_equal(y_true, y_true2):
    print('⚠ Sıralama farklı, McNemar atlanıyor')
    can_mcnemar = False
else:
    can_mcnemar = True

In [ ]:
# Karşılaştırma tablosu
print('\n' + '='*70)
print('CROPPED vs ORIGINAL — Test seti metrikleri')
print('='*70)
print(f'{"Metrik":<18} {"Original":>10} {"Cropped":>10} {"Δ":>10}')
print('-' * 50)
for k in ['accuracy', 'bal_accuracy', 'macro_f1', 'mel_recall', 'mel_precision', 'kappa']:
    delta = crop_m[k] - orig_m[k]
    sign = '+' if delta >= 0 else ''
    print(f'{k:<18} {orig_m[k]:>10.4f} {crop_m[k]:>10.4f} {sign}{delta:>9.4f}')

if can_mcnemar:
    correct_orig = (y_pred_orig == y_true).astype(int)
    correct_crop = (y_pred_crop == y_true).astype(int)
    table = np.array([
        [(correct_crop * correct_orig).sum(),         (correct_crop * (1-correct_orig)).sum()],
        [((1-correct_crop) * correct_orig).sum(),     ((1-correct_crop) * (1-correct_orig)).sum()]
    ])
    res = mcnemar(table, exact=False, correction=True)
    print(f'\nMcNemar (Cropped vs Original): χ²={res.statistic:.4f}, p={res.pvalue:.6f}')
    print(f'Anlamlı (α=0.05): {"✅ EVET" if res.pvalue < 0.05 else "❌ Hayır"}')

## Phase 4 — Lightweight YOLO11n-detect (mobile için)

In [ ]:
DETECT_RUN = 'yolo11n_detect_lesion'
DETECT_DIR = f'{YOLO_RUNS}/{DETECT_RUN}'

# data.yaml'i Colab path'lerine göre düzelt
DATA_YAML = f'{LOCAL_ROOT}/dataset_yolo/data_colab.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(f'''# Auto-generated for Colab
train: {IMG_ROOT}/train
val: {IMG_ROOT}/val
test: {IMG_ROOT}/test

nc: 7
names: ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
''')
print(f'data_colab.yaml oluşturuldu')

if os.path.exists(f'{DETECT_DIR}/weights/best.pt'):
    print(f'⏩ Detect model zaten eğitilmiş: {DETECT_DIR}/weights/best.pt')
else:
    print('=' * 60)
    print(f'YOLO11n-detect EĞİTİM BAŞLIYOR (lightweight, mobile için)')
    print('=' * 60)
    
    t0 = time.time()
    model_d = YOLO('yolo11n.pt')  # nano variant — küçük model
    _ = model_d.train(
        data=DATA_YAML,
        epochs=30,             # detect için 30 epoch yeter
        imgsz=384,             # cls ile aynı resolution
        batch=64,
        patience=10,
        device=0,
        amp=True,
        optimizer='AdamW',
        lr0=0.001,
        cos_lr=True,
        mosaic=1.0,
        mixup=0.0,
        project=YOLO_RUNS,
        name=DETECT_RUN,
        exist_ok=True,
    )
    elapsed = (time.time() - t0) / 60
    print(f'\n✅ Detect eğitim tamamlandı: {elapsed:.1f} dk')

In [ ]:
# Detect modelini test seti üzerinde değerlendir (mAP)
from ultralytics import YOLO

detect_model = YOLO(f'{DETECT_DIR}/weights/best.pt')
val_results = detect_model.val(
    data=DATA_YAML,
    imgsz=384, batch=64, device=0,
    save_json=False, verbose=True
)
print(f'\n✅ Detect val sonucu:')
print(f'  mAP@50    : {val_results.box.map50:.4f}')
print(f'  mAP@50-95 : {val_results.box.map:.4f}')
print(f'  Precision : {val_results.box.mp:.4f}')
print(f'  Recall    : {val_results.box.mr:.4f}')

## Phase 5 — FP16 ONNX Export (mobil için)

In [ ]:
import onnx
from onnxconverter_common import float16

def export_fp16(pt_path, output_name, imgsz=384):
    """YOLO .pt → ONNX FP32 → FP16 (mobile-compatible)."""
    print(f'\n--- {output_name} export ---')
    model = YOLO(pt_path)
    onnx_fp32 = model.export(
        format='onnx', imgsz=imgsz, opset=17, simplify=True, dynamic=False
    )
    print(f'  FP32: {os.path.getsize(onnx_fp32)/1e6:.1f} MB')
    
    m32 = onnx.load(onnx_fp32)
    m16 = float16.convert_float_to_float16(m32, keep_io_types=True)
    
    final_path = f'{PROJECT_ROOT}/{output_name}'
    onnx.save(m16, final_path)
    print(f'  FP16: {final_path} ({os.path.getsize(final_path)/1e6:.1f} MB)')
    return final_path

# 1) Cropped CLS (yeni, ana model)
cls_fp16 = export_fp16(
    f'{RUN_DIR}/weights/best.pt',
    'best_cls_cropped_fp16.onnx'
)

# 2) Lightweight detect (mobile için)
detect_fp16 = export_fp16(
    f'{DETECT_DIR}/weights/best.pt',
    'best_detect_lesion_fp16.onnx'
)

print(f'\n✅ İki ONNX da Drive\'da hazır:')
print(f'  CLS: {cls_fp16}')
print(f'  DET: {detect_fp16}')

total = (os.path.getsize(cls_fp16) + os.path.getsize(detect_fp16)) / 1e6
print(f'\n📊 Toplam mobile boyut: {total:.1f} MB (TÜBİTAK ≤25 MB hedefi: {"✅" if total <= 25 else "❌"})')

## Phase 6 — Final Karşılaştırma (Tüm konfigürasyonlar)

In [ ]:
import json

# Önceki sonuçları yükle (varsa)
prev_summary = {}
for jf in ['multimodal_results.json', 'multimodal_v3_results.json']:
    p = f'{PROJECT_ROOT}/{jf}'
    if os.path.exists(p):
        prev_summary[jf] = json.load(open(p))

# Sonuç özeti
summary = {
    'original_cls': {k: float(v) for k, v in orig_m.items()},
    'cropped_cls': {k: float(v) for k, v in crop_m.items()},
    'mcnemar_pvalue': float(res.pvalue) if can_mcnemar else None,
    'mcnemar_chi2': float(res.statistic) if can_mcnemar else None,
    'cls_onnx_size_mb': round(os.path.getsize(cls_fp16)/1e6, 2),
    'detect_onnx_size_mb': round(os.path.getsize(detect_fp16)/1e6, 2),
    'detect_map50': float(val_results.box.map50),
    'detect_recall': float(val_results.box.mr),
}

with open(f'{PROJECT_ROOT}/segmentation_v1_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 80)
print('SEGMENTATION v1 — TÜBİTAK FINAL DEĞERLENDİRME')
print('=' * 80)
TUBITAK = {'accuracy': 0.90, 'macro_f1': 0.85, 'mel_recall': 0.85, 'kappa': 0.85}
print(f'{"Metrik":<15} {"Orig":>10} {"Cropped":>10} {"Hedef":>8}  {"Sonuç":>10}')
print('-' * 70)
for k in ['accuracy', 'macro_f1', 'mel_recall', 'kappa']:
    orig_v = orig_m[k]
    crop_v = crop_m[k]
    target = TUBITAK[k]
    best = max(orig_v, crop_v)
    flag = '✅' if best >= target else '❌'
    print(f'{k:<15} {orig_v:>10.4f} {crop_v:>10.4f} {target:>8.2f}  {flag:>10}')

print(f'\n📁 Sonuçlar: {PROJECT_ROOT}/segmentation_v1_results.json')
print(f'\n🚀 İndir ve mobile entegre et:')
print(f'   1. {cls_fp16}')
print(f'   2. {detect_fp16}')